In [0]:
##md Loading data into bronze from landing


In [0]:
%fs ls 

In [0]:
help()


In [0]:
help(dbutils.fs)

In [0]:
dbutils.fs.help("ls")

In [0]:
%sql
describe external location checkpoint

In [0]:
checkpoint = spark.sql("describe external location `checkpoint`").select('url').collect()[0][0]

display(checkpoint)

In [0]:
raw_stream =(spark.readStream.format("cloudFiles")\
.option("cloudFiles.format", "csv")\
.load("https://unitycatalogstrg.blob.core.windows.net/landing/rawroad/")
.schema('abfss://checkpoint@unitycatalogstrg.dfs.core.windows.net/checkpoint/inferschema))

In [0]:
raw_stream.printSchema()

In [0]:
landing = spark.sql("describe external location `landing`").select('url').collect()[0][0]

display(landing)

In [0]:
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    from pyspark.sql.functions import current_timestamp
   
    schema = StructType([
    StructField("Record_ID",IntegerType()),
    StructField("Count_point_id",IntegerType()),
    StructField("Direction_of_travel",StringType()),
    StructField("Year",IntegerType()),
    StructField("Count_date",StringType()),
    StructField("hour",IntegerType()),
    StructField("Region_id",IntegerType()),
    StructField("Region_name",StringType()),
    StructField("Local_authority_name",StringType()),
    StructField("Road_name",StringType()),
    StructField("Road_Category_ID",IntegerType()),
    StructField("Start_junction_road_name",StringType()),
    StructField("End_junction_road_name",StringType()),
    StructField("Latitude",DoubleType()),
    StructField("Longitude",DoubleType()),
    StructField("Link_length_km",DoubleType()),
    StructField("Pedal_cycles",IntegerType()),
    StructField("Two_wheeled_motor_vehicles",IntegerType()),
    StructField("Cars_and_taxis",IntegerType()),
    StructField("Buses_and_coaches",IntegerType()),
    StructField("LGV_Type",IntegerType()),
    StructField("HGV_Type",IntegerType()),
    StructField("EV_Car",IntegerType()),
    StructField("EV_Bike",IntegerType())
    ])

    rawTraffic_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{checkpoint}/rawTrafficLoad/schemaInfer')
        .option('header','true')
        .schema(schema)
        .load(landing+'/raw_traffic/')
        .withColumn("Extract_Time", current_timestamp()))
    


In [0]:
rawRoads_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{checkpoint}/rawRoadsLoad/schemaInfer')
        .option('header','true')
        .schema(schema)
        .load(landing+'/raw_roads/'))
        

In [0]:
display(
    rawRoads_stream,
    checkpointLocation="abfss://checkpoint@<storage-account-name>.dfs.core.windows.net/rawRoadsLoad/schemaInfer"
)


In [0]:
f"{checkpoint}/rawRoadsLoad/schemaInfer"

In [0]:
df = spark.read.format("csv").option("header", "true").load(f"{checkpoint}/rawRoadsLoad/schemaInfer")
display(df)